In [1]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: Tesla T4
overlay         113G   83M  113G   1% /
total 33G
-rw-r--r-- 1 501 staff  28G Dec 31  2019 train.parquet
-rw-r--r-- 1 501 staff  88M Dec 31  2019 valid_mask.parquet
-rw-r--r-- 1 501 staff 5.3G Dec 31  2019 valid.parquet
overlay         113G   33G   80G  29% /


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU: Runtime -> Change runtime type -> T4')

Mounted at /content/drive
GPU: Tesla T4


In [3]:
%%writefile /content/drive/MyDrive/wnn/lib.py
import os, sys, time, subprocess
import numpy as np, torch, torch.nn as nn, pyarrow.parquet as pq

dev = 'cuda'
L, NF, WARM, SAT, U3 = 20000, 112, 99, 99, 0.6569
ROOT = '/content/wnn_connectome_starterpack/datasets'
CKPT_DIR = '/content/drive/MyDrive/wnn' if os.path.isdir('/content/drive/MyDrive/wnn') else '/content'
UN = np.array([0.1254, 0.2413, 0.4685, 0.6569])
VALMIX = np.array([0.2125, 0.1853, 0.2301, 0.3721])
post = np.zeros(L, bool); post[WARM:] = True

# ---------- модели ----------
class Net(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.gru = nn.GRU(NF, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Net2(nn.Module):                      # то же, но с произвольным числом входов
    def __init__(self, nin, h=128):
        super().__init__()
        self.gru = nn.GRU(nin, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Step(nn.Module):                      # обёртка для экспорта в ONNX (интерфейс как у baseline)
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, features, hidden_0, hidden_1):
        o, h = self.net(features, torch.cat([hidden_0, hidden_1], 0))
        return o, h[0:1], h[1:2]
StepH = Step

# ---------- данные ----------
def build(src, dst, n, seed, with_mask):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]
    assert len(feat) == NF
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (n, L, NF))
    Y = np.zeros((n, L, 2), np.float32); M = np.zeros((n, L), bool) if with_mask else None
    t = time.time()
    for k, i in enumerate(idx):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'] + (['is_scored'] if with_mask else []))
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat])
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if with_mask: M[k] = tb['is_scored'].to_numpy()
        if k % 500 == 0: print(dst, k, f'{time.time() - t:.0f} с', flush=True)
    X.flush(); np.save(f'/content/data/{dst}_Y.npy', Y)
    if with_mask: np.save(f'/content/data/{dst}_M.npy', M)

def fam_of(Y, verbose=True):                # семейство = ближайшая единица к наименьшему ненулевому |t0|
    a = np.abs(np.clip(Y[..., 0], -2, 2)).astype(np.float64)
    mn = np.where(a > 0, a, 9.0).min(1)
    f = np.abs(mn[:, None] - UN[None]).argmin(1)
    if verbose: print('макс. отклонение от единицы:', np.abs(mn - UN[f]).max().round(4), '| классы:', np.bincount(f, minlength=4))
    return f

# ---------- обучение и метрика ----------
def loss_fn(p, y, mask):
    y = y.clamp(-2, 2)
    return (y.abs() * (p - y) ** 2)[mask].mean()

@torch.no_grad()
def evaluate(model, X, Y, M, bs=50, ch=2000):
    model.eval()
    S = {k: torch.zeros(2, 6, dtype=torch.float64, device=dev) for k in ('все строки', 'is_scored')}
    pm = torch.from_numpy(post).to(dev)
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)
        y = torch.from_numpy(np.ascontiguousarray(Y[s:s + bs])).to(dev).clamp(-2, 2)
        m_sc = torch.from_numpy(np.ascontiguousarray(M[s:s + bs])).to(dev)
        h, outs = None, []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); outs.append(o)
        p = torch.cat(outs, 1).clamp(-2, 2)
        for name, mask in (('все строки', pm[None].expand_as(m_sc)), ('is_scored', m_sc & pm[None])):
            for j in (0, 1):
                yy, pp = y[..., j][mask].double(), p[..., j][mask].double(); a = yy.abs()
                S[name][j] += torch.stack([a.sum(), (a*yy).sum(), (a*pp).sum(), (a*yy*yy).sum(), (a*pp*pp).sum(), (a*yy*pp).sum()])
    out = {}
    for k, T in S.items():
        sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
        cov = T[:, 5] / sw - my * mp
        den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
        c = torch.where(den > 1e-16, cov / den.clamp_min(1e-16).sqrt(), torch.zeros_like(cov)).clamp(-1, 1)
        out[k] = (c.mean().item(), c.cpu().numpy().round(3))
    model.train(); return out

def train(model, epochs, bs=32, ch=1000, lr=2e-3, tag='gru'):     # использует глобальные Xtr, Ytr, Xva, Yva, Mva
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    steps_total = epochs * (len(Xtr) // bs) * (L // ch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps_total, pct_start=0.05)
    rng = np.random.default_rng(0); best = -1
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(len(Xtr)); run = []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev)
            h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1)
                loss = loss_fn(o, y[:, t:t + ch], mask)
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
                run.append(loss.item())
        r = evaluate(model, Xva, Yva, Mva)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | '
              f'valid все строки {r["все строки"][0]:.3f} {r["все строки"][1]} | is_scored {r["is_scored"][0]:.3f} {r["is_scored"][1]}', flush=True)
        if r['is_scored'][0] > best:
            best = r['is_scored'][0]; torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_best.pt')
    return model

# ---------- анализ по семействам и валидационной смеси ----------
@torch.no_grad()
def predict(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s + bs]])).to(dev); h = None; o_ = []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

def sums_np(P, Y, M):
    post_m = np.broadcast_to(post, M.shape); out = {}
    for name, m in (('все строки', post_m), ('is_scored', M & post_m)):
        T = np.zeros((2, 6))
        for j in (0, 1):
            y = np.clip(Y[..., j][m], -2, 2).astype(np.float64); p = np.clip(P[..., j][m], -2, 2).astype(np.float64); a = np.abs(y)
            T[j] = [a.sum(), (a*y).sum(), (a*p).sum(), (a*y*y).sum(), (a*p*p).sum(), (a*y*p).sum()]
        out[name] = T
    return out

def corr(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    cov = T[:, 5] / sw - my * mp
    den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
    return float(np.mean(np.where(den > 1e-16, cov / np.sqrt(np.maximum(den, 1e-16)), 0.0)))

def ens_mix(models):                        # скор по валидационной смеси семейств; нужны Xva, Yva, Mva, fam_v, have
    m = VALMIX / have; Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        P = np.mean([predict(mod, Xva, ix) for mod in models], axis=0)
        Sm[f] = sums_np(P, Yva[ix], Mva[ix])
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
mix_score = lambda model: ens_mix([model])

def code3(y):                               # код k в единицах семейства 4; насыщение = ±99
    k = np.round(y / U3).astype(np.int32); k[y >= 2 - 1e-6] = SAT; k[y <= -2 + 1e-6] = -SAT; return k

def wc(y, p, m):                            # взвешенная корреляция по маске m
    y, p = y[m].astype(np.float64), p[m].astype(np.float64); w = np.abs(y); sw = w.sum()
    my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    cov, vy, vp = (w * (y - my) * (p - mp)).sum(), (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
    return cov / np.sqrt(vy * vp) if vp > 1e-12 else 0.0

Overwriting /content/drive/MyDrive/wnn/lib.py


In [5]:
%run -i /content/drive/MyDrive/wnn/lib.py

import numpy as np, pyarrow.parquet as pq, os, subprocess, time

# ---------- новые признаки ----------
def add_features(X112):
    """
    X112: (L, 112) float32, порядок колонок как в документации.
    Возвращает (L, 6) float32:
      0: микроспред  p11 - p0
      1: средний спред mean(p11..p21) - mean(p0..p10)
      2: Book Imbalance (сумма объёмов)
      3: OFI на лучшем уровне
      4: rolling mean спреда, окно 50
      5: rolling mean спреда, окно 200
    Все NaN/первые шаги заполняются 0.
    """
    L = X112.shape[0]
    out = np.zeros((L, 6), np.float32)
    eps = 1e-8

    # --- цены ---
    bid_p = X112[:, 0:11]      # p0..p10
    ask_p = X112[:, 11:22]     # p11..p21
    bid_v = X112[:, 22:33]     # v0..v10
    ask_v = X112[:, 33:44]     # v11..v21

    # 0: микроспред
    micro = ask_p[:, 0] - bid_p[:, 0]              # p11 - p0
    out[:, 0] = micro

    # 1: средний спред
    out[:, 1] = ask_p.mean(1) - bid_p.mean(1)

    # 2: Book Imbalance
    sb, sa = bid_v.sum(1), ask_v.sum(1)
    out[:, 2] = (sb - sa) / (sb + sa + eps)

    # 3: OFI (Cont) на лучшем уровне
    p0, p11 = bid_p[:, 0], ask_p[:, 0]
    v0, v11 = bid_v[:, 0], ask_v[:, 0]
    e_bid = np.zeros(L, np.float32)
    e_ask = np.zeros(L, np.float32)
    # t >= 1
    up   = p0[1:]  > p0[:-1]
    down = p0[1:]  < p0[:-1]
    flat = ~(up | down)
    e_bid[1:] = np.where(up,   v0[1:],
                 np.where(down, -v0[:-1],
                                v0[1:] - v0[:-1]))
    a_up   = p11[1:] < p11[:-1]   # ask улучшился (упал) -> покупка
    a_down = p11[1:] > p11[:-1]
    e_ask[1:] = np.where(a_up,   v11[1:],
                 np.where(a_down, -v11[:-1],
                                  v11[1:] - v11[:-1]))
    out[:, 3] = e_bid - e_ask

    # 4-5: rolling mean микроспреда
    for k, win in ((4, 50), (5, 200)):
        c = np.concatenate([[0.0], np.cumsum(micro, dtype=np.float64)])
        idx = np.arange(L)
        lo = np.maximum(0, idx - win + 1)
        out[:, k] = ((c[idx + 1] - c[lo]) / (idx - lo + 1)).astype(np.float32)

    return out


def build_with_features(src, dst, n, seed, with_mask):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]
    assert len(feat) == NF, f'{len(feat)} != {NF}'
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))

    NFX = NF + 6
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (n, L, NFX))
    Y = np.zeros((n, L, 2), np.float32)
    M = np.zeros((n, L), bool) if with_mask else None

    t = time.time()
    for k, i in enumerate(idx):
        cols = feat + ['t0', 't1'] + (['is_scored'] if with_mask else [])
        tb = pf.read_row_group(int(i), columns=cols)
        X112 = np.column_stack([tb[c].to_numpy() for c in feat]).astype(np.float32)
        X[k, :, :NF] = X112
        X[k, :, NF:] = add_features(X112)
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if with_mask:
            M[k] = tb['is_scored'].to_numpy()
        if k % 500 == 0:
            print(dst, k, f'{time.time() - t:.0f} с', flush=True)

    X.flush()
    np.save(f'/content/data/{dst}_Y.npy', Y)
    if with_mask:
        np.save(f'/content/data/{dst}_M.npy', M)
    print(f'{dst} готов: X {X.shape}, память {X.nbytes/1e9:.2f} ГБ')


# ---------- пересобрать датасеты ----------
need = {'train': (600, 0, False), 'train3k': (3000, 0, False), 'valid300': (300, 1, True)}

# старые файлы несовместимы по числу колонок — удаляем, чтобы build не пропустил
for n in need:
    for suf in ('X', 'Y', 'M'):
        p = f'/content/data/{n}_{suf}.npy'
        if os.path.exists(p):
            os.remove(p)

if not os.path.exists(f'{ROOT}/train.parquet'):
    print('качаю данные...')
    subprocess.run("curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
                   "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null",
                   shell=True, cwd='/content')

for n, args in need.items():
    src = 'valid' if n.startswith('valid') else 'train'
    build_with_features(src, n, *args)

print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

train 0 0 с
train 500 50 с
train готов: X (600, 20000, 118), память 5.66 ГБ
train3k 0 0 с
train3k 500 57 с
train3k 1000 115 с
train3k 1500 178 с
train3k 2000 239 с
train3k 2500 309 с
train3k готов: X (3000, 20000, 118), память 28.32 ГБ
valid300 0 0 с
valid300 готов: X (300, 20000, 118), память 2.83 ГБ
overlay         113G   68G   46G  60% /



In [6]:
import numpy as np
X = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Y = np.load('/content/data/valid300_Y.npy')

micro = X[:, :, 112]
print('микроспред: min %.6f  max %.6f  доля <0: %.4f' % (micro.min(), micro.max(), (micro < 0).mean()))
bi = X[:, :, 114]
print('BI: min %.4f  max %.4f' % (bi.min(), bi.max()))
ofi = X[:, :, 115]
print('OFI: min %.4f  max %.4f  std %.4f' % (ofi.min(), ofi.max(), ofi.std()))

post = slice(99, None)
y0 = np.abs(Y[:, post, 0]).ravel()
y1 = np.abs(Y[:, post, 1]).ravel()
for j, name in enumerate(['micro_spread', 'mean_spread', 'BI', 'OFI', 'roll50', 'roll200']):
    f = X[:, post, 112 + j].ravel()
    c0 = np.corrcoef(f, y0)[0, 1]
    c1 = np.corrcoef(f, y1)[0, 1]
    print(f'{name:12s}  corr |t0| = {c0:+.3f}   corr |t1| = {c1:+.3f}')

микроспред: min -3.829873  max 2.699837  доля <0: 0.0184
BI: min -23338660.0000  max 3073455.0000
OFI: min -4.8197  max 4.7559  std 0.8312
micro_spread  corr |t0| = -0.000   corr |t1| = -0.000
mean_spread   corr |t0| = +0.028   corr |t1| = +0.012
BI            corr |t0| = -0.001   corr |t1| = -0.001
OFI           corr |t0| = +0.027   corr |t1| = -0.061
roll50        corr |t0| = -0.037   corr |t1| = -0.015
roll200       corr |t0| = -0.031   corr |t1| = -0.011


In [7]:
X = np.load('/content/data/valid300_X.npy', mmap_mode='r')
x = X[0, 1000, :112]   # один шаг, одна последовательность
for name, rng in [('i0_p0..10', range(0,11)), ('i0_p11..21', range(11,22)),
                  ('i0_v0..10', range(22,33)), ('i0_v11..21', range(33,44)),
                  ('i1_p0..10', range(52,63)), ('i1_p11..21', range(63,74))]:
    print(name, np.round(x[list(rng)], 4))

i0_p0..10 [-1.2206  0.1708  0.4057 -1.2758 -0.015   0.8122 -0.2967 -0.1142 -0.6342
 -0.3374  1.0262]
i0_p11..21 [-1.1644  0.2544  0.6335 -1.5179 -0.5465  1.6294  0.0444  0.5917  2.3235
  1.2519 -1.4106]
i0_v0..10 [ 0.4534 -0.1399 -0.2999 -0.4401  0.5953  0.4831  0.4071  0.7335  0.5068
 -0.3762  0.2549]
i0_v11..21 [ 0.0424  0.1218  0.4099 -0.1436 -0.3968 -0.3803 -0.4357 -0.4078  0.0751
  0.4305 -0.4263]
i1_p0..10 [-1.7068  0.1116  0.3772 -1.6727  0.0552  0.7508 -0.3802 -0.16   -0.92
 -0.5633  1.2428]
i1_p11..21 [-0.9951  0.3123  0.7188 -1.4316 -0.7737  1.7639  0.0331  0.6063  2.2876
  1.3481 -1.4637]


In [8]:
def add_features(X112):
    L = X112.shape[0]
    out = np.zeros((L, 6), np.float32)
    eps = 1e-8

    bid_p = X112[:, 0:11]      # p0..p10
    ask_p = X112[:, 11:22]     # p11..p21
    bid_v = X112[:, 22:33]     # v0..v10
    ask_v = X112[:, 33:44]     # v11..v21

    bp_m = bid_p.mean(1)       # среднее bid-отклонение
    ap_m = ask_p.mean(1)       # среднее ask-отклонение
    bv_m = bid_v.mean(1)
    av_m = ask_v.mean(1)

    # 0: спред-прокси (разность средних отклонений ask и bid)
    out[:, 0] = ap_m - bp_m
    # 1: сумма средних (перекос стакана в целом)
    out[:, 1] = ap_m + bp_m
    # 2: дисбаланс "объёмов" (разность средних, нормированная)
    out[:, 2] = (bv_m - av_m) / (np.abs(bv_m) + np.abs(av_m) + eps)
    # 3: OFI-прокси (изменение суммы средних p и v)
    d_bp = np.zeros(L, np.float32); d_ap = np.zeros(L, np.float32)
    d_bp[1:] = bp_m[1:] - bp_m[:-1]
    d_ap[1:] = ap_m[1:] - ap_m[:-1]
    out[:, 3] = d_bp - d_ap
    # 4-5: rolling mean спред-прокси
    for k, win in ((4, 50), (5, 200)):
        c = np.concatenate([[0.0], np.cumsum(out[:, 0], dtype=np.float64)])
        idx = np.arange(L); lo = np.maximum(0, idx - win + 1)
        out[:, k] = ((c[idx + 1] - c[lo]) / (idx - lo + 1)).astype(np.float32)

    return out

In [9]:
# пересобрать только valid300 — 30 секунд
build_with_features('valid', 'valid300', 300, 1, True)

valid300 0 0 с
valid300 готов: X (300, 20000, 118), память 2.83 ГБ


In [10]:
import numpy as np
X = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Y = np.load('/content/data/valid300_Y.npy')

print('спред-прокси: min %.4f max %.4f std %.4f' % (X[:,:,112].min(), X[:,:,112].max(), X[:,:,112].std()))
print('дисбаланс:    min %.4f max %.4f' % (X[:,:,114].min(), X[:,:,114].max()))

post = slice(99, None)
y0 = np.abs(Y[:, post, 0]).ravel()
y1 = np.abs(Y[:, post, 1]).ravel()
for j, name in enumerate(['spread_proxy', 'sum_proxy', 'imbalance', 'ofi_proxy', 'roll50', 'roll200']):
    f = X[:, post, 112 + j].ravel()
    print(f'{name:12s}  corr |t0| = {np.corrcoef(f, y0)[0,1]:+.3f}   corr |t1| = {np.corrcoef(f, y1)[0,1]:+.3f}')

спред-прокси: min -0.4018 max 2.1846 std 0.0871
дисбаланс:    min -1.0000 max 1.0000
spread_proxy  corr |t0| = +0.028   corr |t1| = +0.012
sum_proxy     corr |t0| = -0.020   corr |t1| = -0.035
imbalance     corr |t0| = -0.010   corr |t1| = -0.011
ofi_proxy     corr |t0| = -0.032   corr |t1| = -0.042
roll50        corr |t0| = -0.023   corr |t1| = -0.002
roll200       corr |t0| = -0.025   corr |t1| = -0.006


In [12]:
%run -i /content/drive/MyDrive/wnn/lib.py
NF = 118

# дальше — загрузка данных и обучение
Xtr = np.load('/content/data/train_X.npy', mmap_mode='r')
Ytr = np.load('/content/data/train_Y.npy')
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)

for seed in (0, 1):
    torch.manual_seed(seed)
    m = train(Net().to(dev), epochs=8, tag=f'feat_{seed}')
    print('>>> признаки, сид', seed, mix_score(m), flush=True)

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]


/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/8 | 32 с | loss 1.1370 | valid все строки 0.438 [0.443 0.434] | is_scored 0.632 [0.632 0.633]
эпоха 2/8 | 23 с | loss 1.0612 | valid все строки 0.455 [0.456 0.453] | is_scored 0.631 [0.628 0.634]
эпоха 3/8 | 32 с | loss 1.0359 | valid все строки 0.456 [0.464 0.448] | is_scored 0.642 [0.645 0.638]
эпоха 4/8 | 36 с | loss 1.0220 | valid все строки 0.458 [0.457 0.458] | is_scored 0.637 [0.641 0.632]
эпоха 5/8 | 37 с | loss 1.0010 | valid все строки 0.466 [0.467 0.465] | is_scored 0.645 [0.649 0.641]
эпоха 6/8 | 38 с | loss 0.9811 | valid все строки 0.468 [0.472 0.464] | is_scored 0.656 [0.651 0.661]
эпоха 7/8 | 40 с | loss 0.9547 | valid все строки 0.471 [0.47  0.471] | is_scored 0.655 [0.65  0.661]
эпоха 8/8 | 38 с | loss 0.9491 | valid все строки 0.469 [0.469 0.47 ] | is_scored 0.656 [0.65  0.662]
>>> признаки, сид 0 {'все строки': 0.47, 'is_scored': 0.643}
эпоха 1/8 | 39 с | loss 1.1396 | valid все строки 0.440 [0.443 0.438] | is_scored 0.633 [0.629 0.636]
эпоха 2/8 | 43 с | lo

In [13]:
def make_loss(lam):
    def f(p, y, mask):
        y = y.clamp(-2, 2); main = (y.abs() * (p - y)**2)[mask].mean()
        cs = []
        for j in (0, 1):
            yy, pp = y[..., j][mask], p[..., j][mask]; w = yy.abs(); sw = w.sum() + 1e-8
            my, mp = (w * yy).sum() / sw, (w * pp).sum() / sw
            cov = (w * (yy - my) * (pp - mp)).sum()
            vy, vp = (w * (yy - my)**2).sum(), (w * (pp - mp)**2).sum()
            cs.append(cov / torch.sqrt(vy * vp + 1e-8))
        return main - lam * torch.stack(cs).mean()
    return f

loss_fn = make_loss(0.5)
for seed in (0, 1):
    torch.manual_seed(seed)
    m = train(Net().to(dev), epochs=8, tag=f'feat_corr_{seed}')
    print('>>> признаки + корреляция, сид', seed, mix_score(m), flush=True)
loss_fn = loss_orig

эпоха 1/8 | 44 с | loss 0.9744 | valid все строки 0.443 [0.445 0.441] | is_scored 0.629 [0.629 0.63 ]
эпоха 2/8 | 41 с | loss 0.8655 | valid все строки 0.456 [0.46  0.451] | is_scored 0.639 [0.639 0.639]
эпоха 3/8 | 42 с | loss 0.8366 | valid все строки 0.459 [0.465 0.453] | is_scored 0.642 [0.644 0.64 ]
эпоха 4/8 | 42 с | loss 0.8134 | valid все строки 0.455 [0.46  0.451] | is_scored 0.635 [0.643 0.626]
эпоха 5/8 | 43 с | loss 0.7851 | valid все строки 0.472 [0.472 0.471] | is_scored 0.649 [0.649 0.649]
эпоха 6/8 | 41 с | loss 0.7595 | valid все строки 0.470 [0.474 0.466] | is_scored 0.658 [0.651 0.665]
эпоха 7/8 | 42 с | loss 0.7268 | valid все строки 0.474 [0.473 0.475] | is_scored 0.659 [0.653 0.666]
эпоха 8/8 | 41 с | loss 0.7208 | valid все строки 0.473 [0.472 0.474] | is_scored 0.661 [0.654 0.668]
>>> признаки + корреляция, сид 0 {'все строки': 0.473, 'is_scored': 0.649}
эпоха 1/8 | 39 с | loss 0.9741 | valid все строки 0.442 [0.442 0.441] | is_scored 0.643 [0.639 0.646]
эпоха 2

NameError: name 'loss_orig' is not defined

In [14]:
import numpy as np
Y = np.load('/content/data/valid300_Y.npy')   # (300, 20000, 2)
post = slice(99, None)

for lag in [1, 2, 5, 10, 50, 100, 500, 1000]:
    # сдвиг внутри каждой последовательности, без перетекания между ними
    a = Y[:, post, 0][:, :-lag].ravel()
    b = Y[:, post, 0][:, lag:].ravel()
    c = np.corrcoef(a, b)[0, 1]
    print(f't0 автокорр lag={lag:5d}: {c:+.4f}')

# кросс-корреляция t0 и t1
a = Y[:, post, 0].ravel(); b = Y[:, post, 1].ravel()
print(f't0 vs t1 (одновременно): {np.corrcoef(a, b)[0,1]:+.4f}')
for lag in [1, 5, 50, 500]:
    a = Y[:, post, 1][:, :-lag].ravel()
    b = Y[:, post, 0][:, lag:].ravel()
    print(f't1[t] vs t0[t+{lag}]: {np.corrcoef(a, b)[0,1]:+.4f}')

t0 автокорр lag=    1: +0.9715
t0 автокорр lag=    2: +0.9491
t0 автокорр lag=    5: +0.9067
t0 автокорр lag=   10: +0.8786
t0 автокорр lag=   50: +0.6536
t0 автокорр lag=  100: +0.4795
t0 автокорр lag=  500: +0.1299
t0 автокорр lag= 1000: +0.0568
t0 vs t1 (одновременно): -0.6608
t1[t] vs t0[t+1]: -0.6515
t1[t] vs t0[t+5]: -0.6078
t1[t] vs t0[t+50]: -0.4324
t1[t] vs t0[t+500]: -0.0468


In [15]:
M = np.load('/content/data/valid300_M.npy')   # (300, 20000) bool
print('доля is_scored по шагам (после WARM):', M[:, 99:].mean())
# по семействам
fam_v = fam_of(Y, verbose=False)
for f in range(4):
    print(f'семейство {f}: доля is_scored = {M[fam_v==f, 99:].mean():.3f}')
# временная структура: кластеры?
for i in range(3):
    m = M[i, 99:]
    runs = np.diff(np.concatenate([[0], m.astype(int), [0]]))
    starts = np.where(runs == 1)[0]; ends = np.where(runs == -1)[0]
    print(f'seq {i}: {len(starts)} кластеров, средняя длина {np.mean(ends-starts):.1f}')
# корреляция is_scored с |t0|
a = M[:, 99:].ravel(); b = np.abs(Y[:, 99:, 0]).ravel()
print(f'corr(is_scored, |t0|) = {np.corrcoef(a, b)[0,1]:+.4f}')

доля is_scored по шагам (после WARM): 0.0947541999564511
семейство 0: доля is_scored = 0.137
семейство 1: доля is_scored = 0.080
семейство 2: доля is_scored = 0.034
семейство 3: доля is_scored = 0.123
seq 0: 1265 кластеров, средняя длина 3.8
seq 1: 944 кластеров, средняя длина 3.3
seq 2: 719 кластеров, средняя длина 3.5
corr(is_scored, |t0|) = +0.0463


In [16]:
X = np.load('/content/data/valid300_X.npy', mmap_mode='r')
# статистика по каждому из 112 признаков
stats = []
for j in range(112):
    col = X[:, 99:, j].ravel()
    stats.append((j, col.min(), col.max(), col.mean(), col.std(), np.isnan(col).mean(), np.isinf(col).mean()))
stats = np.array(stats)
print('признаки с NaN:', stats[stats[:,5] > 0][:,0])
print('признаки с inf:', stats[stats[:,6] > 0][:,0])
print('признаки с std=0:', stats[stats[:,4] < 1e-8][:,0])
# топ-10 по std (самые «активные»)
top = stats[np.argsort(-stats[:,4])[:10]]
for row in top:
    print(f'  col {int(row[0]):3d}: min {row[1]:+.3f} max {row[2]:+.3f} std {row[4]:.3f}')

признаки с NaN: []
признаки с inf: []
признаки с std=0: []
  col 111: min -5.199 max +2.757 std 2.931
  col 110: min -5.199 max +2.940 std 2.112
  col 109: min -5.199 max +2.793 std 1.883
  col 105: min -5.199 max +5.199 std 0.972
  col 108: min -3.395 max +5.199 std 0.932
  col 104: min -5.199 max +2.581 std 0.925
  col  87: min -2.419 max +2.651 std 0.908
  col  81: min -2.405 max +2.438 std 0.900
  col  80: min -2.409 max +2.471 std 0.893
  col 103: min -5.199 max +5.199 std 0.868


In [17]:
import pyarrow.parquet as pq
pf = pq.ParquetFile('/content/wnn_connectome_starterpack/datasets/valid.parquet')
tb = pf.read_row_group(0, columns=['seq_ix', 'step_in_seq'])
print('seq_ix уникальных:', tb['seq_ix'].unique().to_pylist()[:10])
print('step_in_seq от', tb['step_in_seq'][0].as_py(), 'до', tb['step_in_seq'][-1].as_py())
# проверьте: идут ли последовательности подряд по seq_ix?
all_seq = set()
for i in range(min(50, pf.num_row_groups)):
    t = pf.read_row_group(i, columns=['seq_ix'])
    all_seq.update(t['seq_ix'].unique().to_pylist())
print('уникальных seq_ix в первых 50 row groups:', len(all_seq), sorted(all_seq)[:10])

seq_ix уникальных: [256678]
step_in_seq от 0 до 19999
уникальных seq_ix в первых 50 row groups: 50 [256678, 751005, 1200028, 1892371, 1990332, 2308749, 4490992, 6746551, 8348957, 9147005]


In [18]:
# взять две соседние последовательности и посмотреть, стыкуются ли признаки
X = np.load('/content/data/valid300_X.npy', mmap_mode='r')
# если seq_ix подряд, то X[0, -1] и X[1, 0] должны быть близки по p0, p11
print('конец seq 0 (p0, p11):', X[0, -1, 0], X[0, -1, 11])
print('начало seq 1 (p0, p11):', X[1, 0, 0], X[1, 0, 11])
print('разница:', abs(X[0, -1, 0] - X[1, 0, 0]), abs(X[0, -1, 11] - X[1, 0, 11]))

конец seq 0 (p0, p11): -2.0366898 -1.1433045
начало seq 1 (p0, p11): -1.6399763 -0.8221767
разница: 0.3967135 0.32112777


In [20]:
for j in range(104, 112):
    col = X[:, 99:, j].ravel()
    print(f'a{j-104}: min {col.min():+.3f} max {col.max():+.3f} mean {col.mean():+.3f} std {col.std():.3f} uniq {len(np.unique(col))}')

a0: min -5.199 max +2.581 mean -0.038 std 0.925 uniq 975960
a1: min -5.199 max +5.199 mean -0.001 std 0.972 uniq 3602115
a2: min -3.073 max +2.411 mean -0.188 std 0.833 uniq 5678479
a3: min -3.122 max +2.380 mean -0.283 std 0.857 uniq 5640923
a4: min -3.395 max +5.199 mean +0.077 std 0.932 uniq 4496940
a5: min -5.199 max +2.793 mean -0.288 std 1.883 uniq 3333330
a6: min -5.199 max +2.940 mean -0.748 std 2.112 uniq 2930573
a7: min -5.199 max +2.757 mean -1.958 std 2.931 uniq 145


In [21]:
def loss_fn(p, y, mask):
    y = y.clamp(-2, 2); p = p.clamp(-2, 2)
    main = (y.abs() * (p - y)**2)[mask].mean()
    # штраф за несогласованность: (p0 + p1)^2 мало, т.к. t0 и t1 антикоррелированы
    consistency = ((p[..., 0] + p[..., 1])**2)[mask].mean()
    return main + 0.1 * consistency

In [22]:
# в add_features или отдельно: сдвиг таргета
Y = np.load('/content/data/train_Y.npy')
X = np.load('/content/data/train_X.npy', mmap_mode='r')
# новый X: 112 + 2 (t0[t-1], t1[t-1])
Xnew = np.lib.format.open_memmap('/content/data/train_ar.npy', 'w+', np.float32, (len(X), L, 114))
Xnew[:, :, :112] = X[:, :, :112]
Xnew[:, 1:, 112] = Y[:, :-1, 0]
Xnew[:, 1:, 113] = Y[:, :-1, 1]
Xnew[:, 0, 112:] = 0   # первый шаг — нули
Xnew.flush()

In [24]:
%run -i /content/drive/MyDrive/wnn/lib.py

# пересобрать чистые 112-колоночные версии
build('train', 'train', 600, 0, False)
build('valid', 'valid300', 300, 1, True)

Xtr = np.load('/content/data/train_X.npy', mmap_mode='r')     # 112
Ytr = np.load('/content/data/train_Y.npy')
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')  # 112
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)

def loss_cons(p, y, mask):
    y = y.clamp(-2, 2); p = p.clamp(-2, 2)
    main = (y.abs() * (p - y)**2)[mask].mean()
    cons = ((p[..., 0] + p[..., 1])**2)[mask].mean()
    return main + 0.1 * cons

loss_fn = loss_cons
for seed in (0, 1):
    torch.manual_seed(seed)
    m = train(Net().to(dev), epochs=8, tag=f'cons_{seed}')
    print('>>> штраф несогласованности, сид', seed, mix_score(m), flush=True)

train 0 0 с
train 500 66 с
valid300 0 0 с
макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
эпоха 1/8 | 39 с | loss 1.2143 | valid все строки 0.423 [0.421 0.424] | is_scored 0.637 [0.63  0.643]
эпоха 2/8 | 37 с | loss 1.1349 | valid все строки 0.448 [0.454 0.441] | is_scored 0.645 [0.645 0.645]
эпоха 3/8 | 35 с | loss 1.1052 | valid все строки 0.442 [0.448 0.435] | is_scored 0.643 [0.643 0.644]
эпоха 4/8 | 31 с | loss 1.0928 | valid все строки 0.439 [0.42  0.458] | is_scored 0.640 [0.63 0.65]
эпоха 5/8 | 29 с | loss 1.0670 | valid все строки 0.453 [0.442 0.464] | is_scored 0.642 [0.63  0.655]
эпоха 6/8 | 23 с | loss 1.0480 | valid все строки 0.453 [0.458 0.448] | is_scored 0.650 [0.643 0.657]
эпоха 7/8 | 19 с | loss 1.0194 | valid все строки 0.453 [0.451 0.454] | is_scored 0.649 [0.641 0.658]
эпоха 8/8 | 21 с | loss 1.0122 | valid все строки 0.452 [0.45  0.454] | is_scored 0.650 [0.641 0.658]
>>> штраф несогласованности, сид 0 {'все строки': 0.451, 'is_scored': 0.637}
эпоха 1/8 

In [25]:
# autoregressive inference: подавать свои предсказания
def predict_ar(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s+bs]])).to(dev)
        # обнулить каналы 112, 113 (они будут заполняться предсказаниями)
        x[:, :, 112:] = 0
        h = None; o_ = []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t+ch], h)
            o_.append(o)
            # заполнить следующие шаги своими предсказаниями
            if t + ch < L:
                x[:, t+1:t+ch+1, 112] = o[:, :-1, 0].detach()
                x[:, t+1:t+ch+1, 113] = o[:, :-1, 1].detach()
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

In [27]:
import numpy as np

Yva_raw = np.load('/content/data/valid300_Y.npy')                 # (300, 20000, 2)
Xva_raw = np.load('/content/data/valid300_X.npy', mmap_mode='r')  # (300, 20000, 112)

Xva_ar = np.lib.format.open_memmap('/content/data/valid300_ar.npy', 'w+',
                                   np.float32, (len(Xva_raw), L, 114))
Xva_ar[:, :, :112] = Xva_raw[:, :, :112]
Xva_ar[:, 1:, 112] = Yva_raw[:, :-1, 0]
Xva_ar[:, 1:, 113] = Yva_raw[:, :-1, 1]
Xva_ar[:, 0, 112:] = 0
Xva_ar.flush()
print('valid300_ar:', Xva_ar.shape, f'{Xva_ar.nbytes/1e9:.2f} ГБ')

valid300_ar: (300, 20000, 114) 2.74 ГБ


In [30]:
import numpy as np
for name in ['train_X', 'valid300_X', 'train_ar', 'valid300_ar', 'train_Y', 'valid300_Y', 'valid300_M']:
    try:
        a = np.load(f'/content/data/{name}.npy', mmap_mode='r')
        print(f'{name:14s} {a.shape}  dtype={a.dtype}')
    except FileNotFoundError:
        print(f'{name:14s} НЕТ ФАЙЛА')

train_X        (600, 20000, 112)  dtype=float32
valid300_X     (300, 20000, 112)  dtype=float32
train_ar       (600, 20000, 114)  dtype=float32
valid300_ar    (300, 20000, 114)  dtype=float32
train_Y        (600, 20000, 2)  dtype=float32
valid300_Y     (300, 20000, 2)  dtype=float32
valid300_M     (300, 20000)  dtype=bool


In [34]:
@torch.no_grad()
def predict_ar(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s+bs]])).to(dev).clone()
        x[:, :, 112:] = 0
        h = None; o_ = []
        for t in range(0, L, ch):
            t_end = min(t + ch, L)
            o, h = model(x[:, t:t_end], h); h = h.detach()
            o_.append(o)
            # если есть следующий шаг — заполнить его своим предсказанием
            if t_end < L:
                n = t_end - t                  # длина текущего чанка
                x[:, t+1:t_end, 112] = o[:, :-1, 0].detach()   # шаги t+1 .. t_end-1
                x[:, t+1:t_end, 113] = o[:, :-1, 1].detach()
                # первый шаг следующего чанка = последнее предсказание текущего
                x[:, t_end, 112] = o[:, -1, 0].detach()
                x[:, t_end, 113] = o[:, -1, 1].detach()
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

In [35]:
%run -i /content/drive/MyDrive/wnn/lib.py
import numpy as np, torch

Xva = np.load('/content/data/valid300_ar.npy', mmap_mode='r')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)

ar = Net2(114).to(dev)
ar.load_state_dict(torch.load(f'{CKPT_DIR}/ar_probe_best.pt', map_location=dev)); ar.eval()

@torch.no_grad()
def predict_ar(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s+bs]])).to(dev).clone()
        x[:, :, 112:] = 0
        h = None; o_ = []
        for t in range(0, L, ch):
            t_end = min(t + ch, L)
            o, h = model(x[:, t:t_end], h); h = h.detach()
            o_.append(o)
            if t_end < L:
                x[:, t+1:t_end, 112] = o[:, :-1, 0].detach()
                x[:, t+1:t_end, 113] = o[:, :-1, 1].detach()
                x[:, t_end, 112] = o[:, -1, 0].detach()
                x[:, t_end, 113] = o[:, -1, 1].detach()
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

def ens_mix_ar(model):
    m = VALMIX / have; Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        P = predict_ar(model, Xva, ix)
        Sm[f] = sums_np(P, Yva[ix], Mva[ix])
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}

print('autoregressive inference (без утечки):', ens_mix_ar(ar))
print('teacher forcing (с утечкой):', mix_score(ar))

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
autoregressive inference (без утечки): {'все строки': 0.005, 'is_scored': -0.134}
teacher forcing (с утечкой): {'все строки': 0.991, 'is_scored': 0.993}


In [36]:
%run -i /content/drive/MyDrive/wnn/lib.py
NF = 112

# пересобрать train3k (112 колонок, без LOB-признаков) — 4 минуты
build('train', 'train3k', 3000, 0, False)

Xtr = np.load('/content/data/train3k_X.npy', mmap_mode='r')
Ytr = np.load('/content/data/train3k_Y.npy')
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)

def make_loss(lam):
    def f(p, y, mask):
        y = y.clamp(-2, 2); main = (y.abs() * (p - y)**2)[mask].mean()
        cs = []
        for j in (0, 1):
            yy, pp = y[..., j][mask], p[..., j][mask]; w = yy.abs(); sw = w.sum() + 1e-8
            my, mp = (w * yy).sum() / sw, (w * pp).sum() / sw
            cov = (w * (yy - my) * (pp - mp)).sum()
            vy, vp = (w * (yy - my)**2).sum(), (w * (pp - mp)**2).sum()
            cs.append(cov / torch.sqrt(vy * vp + 1e-8))
        return main - lam * torch.stack(cs).mean()
    return f

loss_fn = make_loss(0.5)
for seed in (0, 1):
    torch.manual_seed(seed)
    m = train(Net().to(dev), epochs=8, tag=f'corr3k_{seed}')
    print('>>> 3000 + корреляция, сид', seed, mix_score(m), flush=True)

train3k 0 0 с
train3k 500 72 с
train3k 1000 138 с
train3k 1500 206 с
train3k 2000 275 с
train3k 2500 356 с
макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
эпоха 1/8 | 157 с | loss 0.8749 | valid все строки 0.434 [0.428 0.44 ] | is_scored 0.648 [0.64  0.655]
эпоха 2/8 | 153 с | loss 0.8041 | valid все строки 0.472 [0.472 0.473] | is_scored 0.652 [0.648 0.655]
эпоха 3/8 | 152 с | loss 0.7891 | valid все строки 0.476 [0.48  0.473] | is_scored 0.651 [0.653 0.65 ]
эпоха 4/8 | 153 с | loss 0.7731 | valid все строки 0.481 [0.478 0.483] | is_scored 0.662 [0.657 0.668]
эпоха 5/8 | 153 с | loss 0.7576 | valid все строки 0.488 [0.489 0.487] | is_scored 0.661 [0.658 0.664]
эпоха 6/8 | 153 с | loss 0.7317 | valid все строки 0.488 [0.489 0.487] | is_scored 0.665 [0.66  0.669]
эпоха 7/8 | 153 с | loss 0.7057 | valid все строки 0.487 [0.487 0.486] | is_scored 0.665 [0.661 0.669]
эпоха 8/8 | 153 с | loss 0.6914 | valid все строки 0.487 [0.486 0.488] | is_scored 0.665 [0.661 0.67 ]
>>> 3000 + к

In [38]:
%run -i /content/drive/MyDrive/wnn/lib.py
# post восстановлен как np.ndarray

best = Net().to(dev)
best.load_state_dict(torch.load(f'{CKPT_DIR}/corr3k_1_best.pt', map_location=dev)); best.eval()

Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Yva = np.load('/content/data/valid300_Y.npy')
Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)

P = predict(best, Xva, np.arange(len(Xva)))

# --- ПРАВИЛЬНЫЙ диапазон alpha: 1.0 .. 2.5, потому что std(p) < std(y) ---
print('\n=== alpha > 1.0 (растягиваем предсказания) ===')
for alpha in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.8, 2.0, 2.5]:
    P_scaled = np.clip(P * alpha, -2, 2)
    Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        Sm[f] = sums_np(P_scaled[ix], Yva[ix], Mva[ix])
    m = VALMIX / have
    s = {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
    print(f'alpha = {alpha:.1f}: {s}')

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]

=== alpha > 1.0 (растягиваем предсказания) ===
alpha = 1.0: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.1: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.2: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.3: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.4: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.5: {'все строки': 0.49, 'is_scored': 0.654}
alpha = 1.6: {'все строки': 0.489, 'is_scored': 0.653}
alpha = 1.8: {'все строки': 0.488, 'is_scored': 0.651}
alpha = 2.0: {'все строки': 0.485, 'is_scored': 0.649}
alpha = 2.5: {'все строки': 0.479, 'is_scored': 0.642}


In [41]:
!pip install -q onnx onnxruntime

In [42]:
%run -i /content/drive/MyDrive/wnn/lib.py
import torch, os, shutil, importlib.util
from types import SimpleNamespace

# загрузить лучший чекпойнт НА CPU
net = Net(128)                          # без .to(dev)
net.load_state_dict(torch.load(f'{CKPT_DIR}/corr3k_1_best.pt', map_location='cpu'))
net.eval()

os.makedirs('/content/sub_gru', exist_ok=True)
z = torch.zeros(1, 1, 128)              # CPU
torch.onnx.export(
    Step(net),                          # модель на CPU
    (torch.zeros(1, 1, 112), z, z.clone()),
    '/content/sub_gru/model.onnx',
    input_names=['features', 'hidden_0', 'hidden_1'],
    output_names=['prediction', 'hidden_0_out', 'hidden_1_out'],
    opset_version=17, dynamo=False
)

# solution.py
src = open('/content/wnn_connectome_starterpack/baseline/solution.py').read()
src = src.replace('baseline.onnx', 'model.onnx')
src = src.replace('return np.asarray(prediction[0, 0], dtype=np.float32)',
                  'return np.nan_to_num(np.asarray(prediction[0, 0], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)')
assert 'nan_to_num' in src
open('/content/sub_gru/solution.py', 'w').write(src)

!cd /content/sub_gru && rm -f submission.zip && zip -q submission.zip solution.py model.onnx && ls -lh

/tmp/ipykernel_4114/2012210613.py:12: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


total 1.5M
-rw-r--r-- 1 root root 755K Nov  4 19:52 model.onnx
-rw-r--r-- 1 root root 3.0K Nov  4 19:52 solution.py
-rw-r--r-- 1 root root 700K Nov  4 19:52 submission.zip


отправлял друг PENDING
→
CHECK
→
SCORE
Duration
28m6s
Memory
624MB
21:19 2026-11-04

1jppox92

baseline_x03.zip
Andrew_Prib
edit
No comment
0/100
0.65045

Еще друг сказал вот это, не знаб юыло у нас это или нет.
baseline_x03 — это бейзлайн организаторов, у которого все прогнозы умножены на 0.3. Больше в нём ничего не изменено: ни модель, ни веса.

Зачем умножать. Метрика обрезает прогнозы до диапазона ±2, а у бейзлайна прогнозы слишком «размашистые»: разброс около 1.8 при разбросе таргетов около 0.9. Из-за этого 40% прогнозов на скорируемых строках упираются в границу и теряют информацию. После умножения на 0.3 обрезка почти не срабатывает, и скор на валидации вырос с 0.631 до 0.686, одинаково на обеих половинах выборки.

Где лежит: хакатон/submissions/baseline_x03.zip. Внутри исходный baseline.onnx и solution.py, отличающийся одной строкой с умножением на 0.3. Я его собрал и проверил харнессом, архив готов к отправке как есть.

Почему его стоит отправить сейчас. Локально он сильнее нашей GRU: 0.686 против 0.659. Значит, на лидерборде он, скорее всего, даст больше ваших 0.644, ориентировочно около 0.67. К тому же это эталон, с которым будем сравнивать дообученную модель

In [44]:
# скор на valid300 через evaluate (по всем строкам, без весов семейств)
net = Net(128).to(dev)
net.load_state_dict(torch.load(f'{CKPT_DIR}/corr3k_1_best.pt', map_location=dev)); net.eval()
r = evaluate(net, Xva, Yva, Mva)
print('evaluate (прямой):', r)
print('mix_score (семейства):', mix_score(net))

evaluate (прямой): {'все строки': (0.4902063049993442, array([0.492, 0.488])), 'is_scored': (0.6664275877194847, array([0.663, 0.669]))}
mix_score (семейства): {'все строки': 0.49, 'is_scored': 0.654}
